# Configuration

In [1]:
import os 

if True ^ os.getcwd().endswith('hte-and-targeting"'):
    os.chdir('..')

In [13]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

import statsmodels.api as sm
from statsmodels.regression.linear_model import OLS

In [3]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# visualization params
# label size
tick_label_size = 12
legend_label_size = 12
axis_label_size = 14
title_size = 18
# font
plt.rcParams['font.family'] = 'serif'

# model label map 
model_label_map = {
    'plugin': 'Plugin', 
    'standard_bootstrap': 'Standard Bootstrap', 
    'mn_bootstrap': 'm-out-of-n Bootstrap', 
    'num_bootstrap': 'Numerical Bootstrap'
}

In [4]:
from core.variables import *
from core.dgp import SingleSegmentTreatmentSelection as SingleSegment
from core.dgp import MultipleSegmentsTreatmentSelection as MultipleSegments 
from core.dgp import ContinuousSegments

import core.treatment_selection_single_segment as tsss
import core.treatment_selection_multiple_segments as tsms 
import core.continuous_segment as cs 

# DGP

In [5]:
# global parameters
delta_tau = 0.1



# treatment selection multiple segments
tsms_dgp_params = {
    'base_te_arr': np.array([1, 1 + delta_tau]), 'n_segments': 5,
    'response_type': 'continuous', 'noise_std': 1,
}
tsms_dgp = MultipleSegments(**tsms_dgp_params)

# Point Estimates vs. Posterior Distribution

## Single Segment

In [44]:
def estimate_te_with_dstn(
    treatments: np.ndarray, outcomes: np.ndarray, response_type: str 
) -> list: 
    """ 
    Estimate treatment effect together with the sampling/posterior distribution of the estimators
    """
    if response_type == 'continuous':
        ols = OLS(endog=outcomes, exog=np.eye(2)[treatments]).fit()
        te_var_list = [UnivariateGaussian(mean=mean, std=std) for mean, std in zip(ols.params, ols.bse)]
    elif response_type == 'logit':
        logit = sm.Logit(endog=outcomes, exog=np.eye(2)[treatments]).fit(disp=0)
        te_var_list = [UnivariateGaussian(mean=mean, std=std) for mean, std in zip(logit.params, logit.bse)]
    else: 
        raise ValueError(f"Unsupport response type: {response_type}")
    
    return te_var_list 

def optimize_with_dstn(
    te_dstns: list, n_draws: int 
) -> tuple:
    """ 
    Optimize targeting decision with the sampling/posterior distribution of the estimators
    """
    # te_draws_arr has shape (n_draws, n_treatments)
    te_draws_arr = np.concatenate([te_dstn.sample(n_draws).reshape(-1, 1) for te_dstn in te_dstns], axis=1)

    # take average (empirical expectation)
    emp_te_arr = te_draws_arr.mean(axis=0)  # shape = (n_treatments,)

    return tsss.optimize(te_arr=emp_te_arr)

In [71]:
# dgp
tsss_dgp_params = {'te_arr': np.array([1, 1.1]), 'noise_std': 1, 'response_type': 'continuous'}
tsss_dgp = SingleSegment(**tsss_dgp_params) 

n_repeats = 5000
sample_size = 100
wc_arr = np.zeros(n_repeats)
wc_dstn_arr = np.zeros(n_repeats)

for i in tqdm(range(n_repeats)):
    # sample data
    treatment_arr, outcome_arr = tsss_dgp.sample(sample_size=sample_size, seed=i)

    # estimate demand model
    emp_te_arr = tsss.estimate_te(treatment_arr, outcome_arr, response_type='continuous')

    # optimize 
    plugin_decision, plugin_val_est = tsss.optimize(te_arr=emp_te_arr)
    true_plugin_val = tsss.obj_func(targ_decision=plugin_decision, te_arr=tsss_dgp.te_arr)

    wc_arr[i] = plugin_val_est - true_plugin_val

    # sample data
    treatment_arr, outcome_arr = tsss_dgp.sample(sample_size=sample_size, seed=i)

    # estimate demand model
    emp_te_arr = tsss.estimate_te(treatment_arr, outcome_arr, response_type='continuous')

    # optimize 
    plugin_decision, plugin_val_est = tsss.optimize(te_arr=emp_te_arr)
    true_plugin_val = tsss.obj_func(targ_decision=plugin_decision, te_arr=tsss_dgp.te_arr)

    # optimzie with distribution
    te_var_list = estimate_te_with_dstn(treatment_arr, outcome_arr, response_type='continuous')
    dstn_plugin_decision, dstn_plugin_val_est = optimize_with_dstn(te_var_list, n_draws=5000)
    true_dstn_plugin_val = tsss.obj_func(targ_decision=dstn_plugin_decision, te_arr=tsss_dgp.te_arr)

    wc_dstn_arr[i] = dstn_plugin_val_est - true_dstn_plugin_val

delta_tau = tsss_dgp.te_arr[1] - tsss_dgp.te_arr[0]
avg_wc = wc_arr.mean()
se_wc = wc_arr.std() / np.sqrt(sample_size)
avg_wc_dstn = wc_dstn_arr.mean()
se_wc_dstn = wc_dstn_arr.std() / np.sqrt(sample_size)

print(f"Average winner's curse using point estiamte: {avg_wc / delta_tau:.2%} ({se_wc / delta_tau:.2%})")
print(f"Average winner's curse using distribution: {avg_wc_dstn / delta_tau:.2%} ({se_wc_dstn / delta_tau:.2%})")

100%|██████████| 5000/5000 [00:03<00:00, 1569.69it/s]

Average winner's curse using point estiamte: 71.49% (12.15%)
Average winner's curse using distribution: 71.58% (12.17%)


In [72]:
# dgp
tsss_dgp_params = {'te_arr': np.array([1, 1.1]), 'noise_std': 1, 'response_type': 'logit'}
tsss_dgp = SingleSegment(**tsss_dgp_params) 

n_repeats = 5000
sample_size = 100
wc_arr = np.zeros(n_repeats)
wc_dstn_arr = np.zeros(n_repeats)

for i in tqdm(range(n_repeats)):
    # sample data
    treatment_arr, outcome_arr = tsss_dgp.sample(sample_size=100)

    # estimate demand model
    emp_te_arr = tsss.estimate_te(treatment_arr, outcome_arr, response_type='logit')

    # optimize 
    plugin_decision, plugin_val_est = tsss.optimize(te_arr=emp_te_arr)
    true_plugin_val = tsss.obj_func(targ_decision=plugin_decision, te_arr=tsss_dgp.te_arr)

    wc_arr[i] = plugin_val_est - true_plugin_val

    # optimzie with distribution
    te_var_list = estimate_te_with_dstn(treatment_arr, outcome_arr, response_type='logit')
    dstn_plugin_decision, dstn_plugin_val_est = optimize_with_dstn(te_var_list, n_draws=10000)
    true_dstn_plugin_val = tsss.obj_func(targ_decision=dstn_plugin_decision, te_arr=tsss_dgp.te_arr)

    wc_dstn_arr[i] = dstn_plugin_val_est - true_dstn_plugin_val

delta_tau = tsss_dgp.te_arr[1] - tsss_dgp.te_arr[0]
avg_wc = wc_arr.mean()
se_wc = wc_arr.std() / np.sqrt(sample_size)
avg_wc_dstn = wc_dstn_arr.mean()
se_wc_dstn = wc_dstn_arr.std() / np.sqrt(sample_size)

print(f"Average winner's curse using point estiamte: {avg_wc / delta_tau:.2%} ({se_wc / delta_tau:.2%})")
print(f"Average winner's curse using distribution: {avg_wc_dstn / delta_tau:.2%} ({se_wc_dstn / delta_tau:.2%})")

100%|██████████| 5000/5000 [00:15<00:00, 316.35it/s]

Average winner's curse using point estiamte: 210.84% (30.48%)
Average winner's curse using distribution: 210.90% (30.52%)
